In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
sys.path.insert(0, str(Path.cwd().parent))

from solvers import (
    euler_explicit, euler_implicit, rk4,
    adams_bashforth_4, adams_moulton_4,
    adams_bashforth_moulton_4,
)
from utils.runge_table import runge_table_system
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#### Модель экономической системы фирмы (Shapovalov)

Система ОДУ:

$$
\dot x = -\sigma x + \delta y \\
\dot y = \mu (x + y) - \beta x z \\
\dot z = -\gamma z + \alpha x y
$$

Исследуем **устойчивость константного решения** (0,0,0) при параметрах из статьи.

In [3]:
def f_firm(t, y, **kwargs):
    """Правая часть модели фирмы"""
    x, y, z = y
    alpha = kwargs.get('alpha', 5.0)
    beta  = kwargs.get('beta', 8.0)
    delta = kwargs.get('delta', 1.0)
    mu    = kwargs.get('mu', 2.1)
    sigma = kwargs.get('sigma', 4.1)
    gamma = kwargs.get('gamma', 1.0)
    return np.array([
        -sigma * x + delta * y,
        mu * (x + y) - beta * x * z,
        -gamma * z + alpha * x * y
    ])

params = {'alpha':5.0, 'beta':8.0, 'delta':1.0, 'mu':2.1, 'sigma':4.1, 'gamma':1.0}
a, b = 0.0, 20.0

O0 = np.array([0.0, 0.0, 0.0])

x_star = np.sqrt(params['gamma'] * params['mu'] * (params['delta'] + params['sigma']) / \
       (params['alpha'] * params['beta'] * params['sigma']))
y_star = np.sqrt(params['gamma'] * params['mu'] * params['sigma'] * (params['delta'] + params['sigma']) /
                 (params['alpha'] * params['beta'] * params['delta']**2))
z_star = params['mu'] * (params['delta'] + params['sigma']) / (params['beta'] * params['delta'])

O1 = np.array([x_star, y_star, z_star])
O2 = np.array([-x_star, -y_star, z_star])

print("O0 =", O0)
print("O1 =", O1)
print("O2 =", O2)

steps = [0.4 * (1/2)**i for i in range(12)]

O0 = [0. 0. 0.]
O1 = [0.25554819 1.04774758 1.33875   ]
O2 = [-0.25554819 -1.04774758  1.33875   ]


In [4]:
np.random.seed(42)

pert = 1e-3 * np.ones(3) + np.random.normal(0, 1)

y0_O0 = O0 + pert
y0_O1 = O1 + pert
y0_O2 = O2 + pert

In [5]:
methods = {
    'Euler explicit':       euler_explicit,
    'Euler implicit':       euler_implicit,
    'RK4':                  rk4,
    'Adams-Bashforth 4':    adams_bashforth_4,
    'Adams-Moulton 4':      adams_moulton_4,
    'ABM 4 (pred-corr)':    adams_bashforth_moulton_4,
}

def build_runge_table_system(methods_dict, bounds, steps, y0, f, etalon_sol=None, **params):
    """Одна большая таблица Рунге для всех методов сразу"""
    frames = []
    for name, meth in methods_dict.items():
        current_etalon = etalon_sol(meth) if etalon_sol else None
        table = runge_table_system(
            bounds=bounds,
            steps=steps,
            y0=y0,
            method=meth,
            f=f,
            name=name,
            exact_solution=None,
            etalon_solution=current_etalon,
            **params
        )
        frames.append(table)
    return pd.concat(frames, axis=1)

In [ ]:
def run_runge_for_equilibrium(eq_name, y0_eq):
    print(f"\n{'='*90}\nТАБЛИЦА РУНГЕ ОКОЛО {eq_name}\n{'='*90}")
    frames = []
    for name, meth in methods.items():
        table = runge_table_system(
            bounds=(a, b),
            steps=steps,
            y0=y0_eq,
            method=meth,
            f=f_firm,
            name=name,
            exact_solution=None,
            etalon_solution=None,
            **params
        )
        frames.append(table)
    big_table = pd.concat(frames, axis=1)
    display(big_table)

run_runge_for_equilibrium("O0 (0,0,0)", y0_O0)
run_runge_for_equilibrium("O1", y0_O1)
run_runge_for_equilibrium("O2", y0_O2)


ТАБЛИЦА РУНГЕ ОКОЛО O0 (0,0,0)


Euler explicit           Euler implicit                     RK4  \
                  Eps_h   Order_p          Eps_h   Order_p         Eps_h   
Step                                                                       
0.400000            NaN       NaN       0.166463 -0.324558  9.104595e+00   
0.200000            NaN       NaN       0.207819 -0.320124  7.516094e-02   
0.100000            NaN       NaN       0.346162 -0.736122  1.787657e-03   
0.050000       4.303010       NaN       3.047623 -3.138163  5.649548e-05   
0.025000       3.656906  0.234723       3.642971 -0.257431  1.828794e-06   
0.012500       3.128188  0.225296       2.731227  0.415566  5.851154e-08   
0.006250       2.025436  0.627095       3.197670 -0.227472  1.852611e-09   
0.003125       3.898413 -0.944655       2.597122  0.300107  5.829343e-11   
0.001563       1.283954  1.602294       1.969177  0.399321  1.828038e-12   
0.000781       0.531713  1.271874       0.531020  1.890754  5.723200e-14   
0.000391       0.183051  1.538399       0.198560  1.419193  1.831868e-15   
0.000195       0.076730  1.254386       0.086203  1.203762  5.551115e-17   

                    Adams-Bashforth 4           Adams-Moulton 4              \
            Order_p             Eps_h   Order_p           Eps_h     Order_p   
Step                                                                          
0.400000  17.356299               NaN       NaN    4.662688e+45  511.600315   
0.200000   6.920468               NaN       NaN    2.432825e+00  150.425293   
0.100000   5.393841               NaN       NaN    5.932522e-01    2.035915   
0.050000   4.983791               NaN       NaN    3.789008e-02    3.968753   
0.025000   4.949171      3.064075e-02       NaN    1.681535e-03    4.493970   
0.012500   4.966027      1.527399e-03  4.326303    1.095747e-04    3.939791   
0.006250   4.981089      1.315830e-04  3.537032    9.777040e-06    3.486374   
0.003125   4.990083      9.798201e-06  3.747312    7.318856e-07    3.739708   
0.001563   4.994965      6.665416e-07  3.877750    5.045762e-08    3.858474   
0.000781   4.997330      4.343533e-08  3.939754    2.875202e-09    4.133337   
0.000391   4.965434      2.771053e-09  3.970363    2.400080e-10    3.582509   
0.000195        NaN      1.752722e-10  3.982765    1.473244e-11    4.026015   

         ABM 4 (pred-corr)            
                     Eps_h   Order_p  
Step                                  
0.400000               NaN       NaN  
0.200000               NaN       NaN  
0.100000      9.925004e-01       NaN  
0.050000      8.144108e-02  3.607239  
0.025000      5.613115e-03  3.858883  
0.012500      2.219838e-04  4.660276  
0.006250      8.206059e-06  4.757621  
0.003125      6.223715e-07  3.720842  
0.001563      4.650664e-08  3.742267  
0.000781      3.160861e-09  3.879047  
0.000391      2.058858e-10  3.940402  
0.000195      1.284350e-11  4.002733


ТАБЛИЦА РУНГЕ ОКОЛО O1


Euler explicit           Euler implicit                     RK4  \
                  Eps_h   Order_p          Eps_h   Order_p         Eps_h   
Step                                                                       
0.400000            NaN       NaN       0.502550 -1.308842  6.799101e-01   
0.200000            NaN       NaN       2.510440 -2.320601  5.442582e-02   
0.100000            NaN       NaN       0.540776  2.214837  6.686535e-03   
0.050000       4.341084       NaN       3.073354 -2.506711  3.266302e-04   
0.025000       4.563724 -0.072156       3.734273 -0.281013  1.201997e-05   
0.012500       2.520410  0.856553       4.120233 -0.141899  4.017643e-07   
0.006250       4.285044 -0.765652       3.735174  0.141551  1.293871e-08   
0.003125       3.316965  0.369446       3.960169 -0.084387  4.101053e-10   
0.001563       3.021807  0.134453       3.202476  0.306374  1.290412e-11   
0.000781       3.929653 -0.378991       3.028863  0.080412  4.045653e-13   
0.000391       1.827891  1.104222       0.711845  2.089140  1.265654e-14   
0.000195       0.460300  1.989533       0.259167  1.457680  4.440892e-16   

                    Adams-Bashforth 4           Adams-Moulton 4              \
            Order_p             Eps_h   Order_p           Eps_h     Order_p   
Step                                                                          
0.400000  12.217968               NaN       NaN    2.135230e+07  500.870488   
0.200000   3.642981               NaN       NaN    2.600715e+00   22.968980   
0.100000   3.024961               NaN       NaN    3.722707e+00   -0.517444   
0.050000   4.355529               NaN       NaN    2.833018e-01    3.715940   
0.025000   4.764153      8.911480e-01       NaN    4.304593e-02    2.718391   
0.012500   4.902940      4.380217e-02  4.346591    3.329379e-03    3.692552   
0.006250   4.956584      2.906833e-03  3.913482    2.247497e-04    3.888862   
0.003125   4.979555      1.901290e-04  3.934398    1.456865e-05    3.947380   
0.001563   4.990090      1.217591e-05  3.964877    9.307344e-07    3.968354   
0.000781   4.995316      7.704526e-07  3.982180    5.796389e-08    4.005143   
0.000391   4.998417      4.845426e-08  3.991011    1.683723e-09    5.105428   
0.000195   4.832890      3.037002e-09  3.995904    1.075764e-10    3.968221   

         ABM 4 (pred-corr)            
                     Eps_h   Order_p  
Step                                  
0.400000               NaN       NaN  
0.200000               NaN       NaN  
0.100000      3.526094e+00       NaN  
0.050000      4.072592e-01  3.114052  
0.025000      2.727810e-02  3.900132  
0.012500      2.277993e-03  3.581908  
0.006250      1.835063e-04  3.633862  
0.003125      1.315945e-05  3.801659  
0.001563      8.816154e-07  3.899806  
0.000781      5.704533e-08  3.949969  
0.000391      3.626941e-09  3.975284  
0.000195      2.288918e-10  3.986016


ТАБЛИЦА РУНГЕ ОКОЛО O2


Euler explicit           Euler implicit                     RK4  \
                  Eps_h   Order_p          Eps_h   Order_p         Eps_h   
Step                                                                       
0.400000            NaN       NaN       0.139637  3.954238  2.493176e+00   
0.200000            NaN       NaN       0.201423 -0.528551  3.880630e-02   
0.100000            NaN       NaN       0.344711 -0.775157  1.061119e-03   
0.050000       4.424635       NaN       3.133439 -3.184288  2.750192e-05   
0.025000       3.398090  0.380834       3.333711 -0.089383  7.225969e-07   
0.012500       3.260262  0.059736       3.995933 -0.261403  1.992803e-08   
0.006250       3.108377  0.068826       3.176335  0.331169  5.765241e-10   
0.003125       3.275729 -0.075654       1.132423  1.487950  1.725342e-11   
0.001563       3.121248  0.069693       2.541631 -1.166342  5.269118e-13   
0.000781       2.552133  0.290419       0.605903  2.068596  1.620926e-14   
0.000391       0.645405  1.983426       0.260875  1.215729  5.551115e-16   
0.000195       0.246574  1.388186       0.158104  0.722482  2.220446e-16   

                    Adams-Bashforth 4           Adams-Moulton 4              \
            Order_p             Eps_h   Order_p           Eps_h     Order_p   
Step                                                                          
0.400000  10.846959               NaN       NaN    5.193926e+23  550.221248   
0.200000   6.005550               NaN       NaN    3.806205e+00   76.852819   
0.100000   5.192633               NaN       NaN    3.295935e+00    0.207665   
0.050000   5.269910               NaN       NaN    2.151787e-01    3.937081   
0.025000   5.250198      3.710747e-02       NaN    8.760709e-03    4.618344   
0.012500   5.180320      2.992686e-03  3.632197    9.720344e-05    6.493896   
0.006250   5.111275      3.895950e-04  2.941394    1.792212e-05    2.439266   
0.003125   5.062426      3.131826e-05  3.636899    1.972834e-06    3.183400   
0.001563   5.033177      2.184682e-06  3.841508    1.563339e-07    3.657567   
0.000781   5.022672      1.437938e-07  3.925350    7.863110e-09    4.313386   
0.000391   4.867896      9.220775e-09  3.962970    5.180629e-10    3.923901   
0.000195   1.321928      5.811893e-10  3.987808    3.275025e-11    3.983550   

         ABM 4 (pred-corr)            
                     Eps_h   Order_p  
Step                                  
0.400000               NaN       NaN  
0.200000               NaN       NaN  
0.100000      1.590126e+00       NaN  
0.050000      4.455627e-01  1.835441  
0.025000      9.691560e-03  5.522756  
0.012500      1.486344e-04  6.026889  
0.006250      2.666986e-05  2.478486  
0.003125      2.310161e-06  3.529145  
0.001563      1.639517e-07  3.816651  
0.000781      1.084846e-08  3.917708  
0.000391      6.941752e-10  3.966047  
0.000195      4.926770e-11  3.816586